<a href="https://colab.research.google.com/github/yuniecorn-dev/Numerai_ESAA/blob/main/%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
2026 신촌 연합 데이터·AI 해커톤 — Bank Churn: PR-AUC 최고 성능 파이프라인
================================================================================
전략: LightGBM + XGBoost를 5-Fold OOF(Out-of-Fold)로 학습 → OOF 예측으로
      최적 블렌드 가중치를 찾고(리키지 없음) → 동일 가중치를 test 예측에 적용.

정직한 벤치마크 (같은 데이터, 여러 방식으로 검증한 결과 — 자세한 과정은 리포트 참고):
    HGB (공식 baseline)                 PR-AUC ≈ 0.7270 (holdout) / 0.7285 (OOF)
    LightGBM 단독                       PR-AUC ≈ 0.7279 (holdout) / 0.7306 (OOF)
    XGBoost 단독                        PR-AUC ≈            -     / 0.7308 (OOF)
    CatBoost 단독                       PR-AUC ≈ 0.7265 (holdout)              <- 가장 낮아 최종에서 제외
    LightGBM+XGBoost 최적 블렌드         PR-AUC ≈            -     / 0.7312 (OOF)  ★ 이 스크립트의 결과물
    (HGB를 블렌드에 추가해도 최적 가중치가 ~0으로 수렴 → 기여 없음, 최종엔 2모델만 사용)

    * 파생변수(Balance==0 플래그, 비율, 연령대 구간 등)와 하이퍼파라미터 튜닝(Optuna 40 trial)은
      모두 시도했지만 raw PR-AUC 기준으로는 유의미한 개선이 없었음 (트리 모델이 이미
      원본 피처만으로 비단조 관계/상호작용을 스스로 학습하기 때문). 그래서 이 최종
      파이프라인은 "원본 피처 + 서로 다른 두 부스팅 알고리즘의 블렌드"에 집중했고,
      이 조합이 지금까지 시도한 것 중 가장 높은 PR-AUC를 냈다.
    * 데이터/모델의 성능 상한 근처이므로 극적인 추가 상승은 기대하기 어렵다는 점은
      보고서의 "한계" 섹션에 정직하게 적는 걸 권장.
"""

**요약**: train.csv로 LightGBM과 XGBoost 두 모델을 각각 5번 나눠 학습시키고(5-fold), 그 예측을 섞어서(블렌딩) 최종 이탈확률을 계산하는 코드예요.

**단계별로 보면**
1. train/test 데이터 불러오기 (원본 11개 피처 그대로 사용)
2. 데이터를 5등분해서, 매번 4/5로 학습 + 1/5로 검증 (5번 반복) — LightGBM과 XGBoost 둘 다
3. 검증 결과(OOF 예측)로 "LightGBM을 몇 %, XGBoost를 몇 % 섞을지" 최적 비율을 찾음 (결과: LightGBM 43% + XGBoost 57%)
4. 같은 비율로 test.csv 예측값도 섞어서 최종 이탈확률 계산
5. `submission_best_stack.csv`로 저장 (Kaggle 제출용)

**한 줄 요약**: "두 모델을 각각 여러 번 학습시켜 검증하고, 가장 좋은 배합 비율을 찾아 섞은 예측값을 제출 파일로 만드는 코드."

In [26]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_score, recall_score, precision_recall_curve)

In [40]:
RANDOM_STATE = 42
N_FOLDS = 5
TARGET = "Exited"
DATA_DIR = "/content/drive/MyDrive/해커톤"
OUT_PATH = "/content/drive/MyDrive/해커톤/submission_best_stack.csv"

NUM_FEATURES = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
                 "HasCrCard", "IsActiveMember", "EstimatedSalary"]
CAT_FEATURES = ["Geography", "Gender"]
FEATURES = NUM_FEATURES + CAT_FEATURES

In [41]:
def load_data():
    train = pd.read_csv(f"{DATA_DIR}/train.csv")
    test = pd.read_csv(f"{DATA_DIR}/test.csv")
    return train, test

In [42]:
def to_categorical(df, ref_categories=None):
    """LightGBM/XGBoost native categorical 처리를 위해 dtype 지정.
    ref_categories가 주어지면(test set) train과 동일한 카테고리 목록으로 맞춘다."""
    df = df.copy()
    for c in CAT_FEATURES:
        cats = ref_categories[c] if ref_categories else pd.Categorical(df[c]).categories
        df[c] = pd.Categorical(df[c], categories=cats)
    return df

In [43]:
def make_lgbm():
    return LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
        min_child_samples=30, random_state=RANDOM_STATE, verbosity=-1,
    )

In [44]:
def make_xgb():
    return XGBClassifier(
        n_estimators=2000, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        tree_method="hist", enable_categorical=True, eval_metric="aucpr",
        early_stopping_rounds=80, random_state=RANDOM_STATE, verbosity=0,
    )

In [45]:
def best_f1_threshold(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
    i = int(np.argmax(f1[:-1])) if len(thr) else 0
    return float(thr[i]), float(f1[i])

In [46]:
def main():
    train, test = load_data()
    X_raw, y = train[FEATURES], train[TARGET]
    Xtest_raw = test[FEATURES]

    ref_cats = {c: pd.Categorical(X_raw[c]).categories for c in CAT_FEATURES}
    Xc = to_categorical(X_raw, ref_cats)
    Xtc = to_categorical(Xtest_raw, ref_cats)

    N, NT = len(train), len(test)
    oof_lgb, oof_xgb = np.zeros(N), np.zeros(N)
    test_lgb, test_xgb = np.zeros(NT), np.zeros(NT)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    print(f"[{N_FOLDS}-Fold OOF 학습 시작] train={N}행, test={NT}행")
    for fold, (tr_i, va_i) in enumerate(skf.split(Xc, y), 1):
        yt, yv = y.iloc[tr_i], y.iloc[va_i]

        lgb = make_lgbm()
        lgb.fit(Xc.iloc[tr_i], yt, eval_set=[(Xc.iloc[va_i], yv)], eval_metric="average_precision",
                callbacks=[early_stopping(80, verbose=False), log_evaluation(0)])
        oof_lgb[va_i] = lgb.predict_proba(Xc.iloc[va_i])[:, 1]
        test_lgb += lgb.predict_proba(Xtc)[:, 1] / N_FOLDS

        xgb = make_xgb()
        xgb.fit(Xc.iloc[tr_i], yt, eval_set=[(Xc.iloc[va_i], yv)], verbose=False)
        oof_xgb[va_i] = xgb.predict_proba(Xc.iloc[va_i])[:, 1]
        test_xgb += xgb.predict_proba(Xtc)[:, 1] / N_FOLDS

        print(f"  fold {fold}: LightGBM PR-AUC={average_precision_score(yv, oof_lgb[va_i]):.4f}  "
              f"XGBoost PR-AUC={average_precision_score(yv, oof_xgb[va_i]):.4f}")

    print(f"\n[개별 모델 OOF 전체] LightGBM={average_precision_score(y, oof_lgb):.4f}  "
          f"XGBoost={average_precision_score(y, oof_xgb):.4f}")

    # ------------------------------------------------------------------
    # 블렌드 가중치 탐색: OOF 예측(=학습에 전혀 안 쓰인 예측)로만 탐색 → 리키지 없음
    # ------------------------------------------------------------------
    best_score, best_w = -1, 0.5
    for w in np.linspace(0, 1, 201):
        s = average_precision_score(y, w * oof_lgb + (1 - w) * oof_xgb)
        if s > best_score:
            best_score, best_w = s, w
    print(f"\n[최적 블렌드] w_LightGBM={best_w:.3f}  w_XGBoost={1-best_w:.3f}")
    print(f"[최종 OOF 성능] ★PR-AUC={best_score:.4f}(공식지표)  "
          f"ROC-AUC={roc_auc_score(y, best_w*oof_lgb+(1-best_w)*oof_xgb):.4f}(참고)")

    oof_blend = best_w * oof_lgb + (1 - best_w) * oof_xgb
    pred05 = (oof_blend >= 0.5).astype(int)
    t_best, f1_best = best_f1_threshold(y, oof_blend)
    pred_b = (oof_blend >= t_best).astype(int)
    print(f"   thr=0.50        F1={f1_score(y, pred05):.4f}  P={precision_score(y, pred05):.4f}  R={recall_score(y, pred05):.4f}")
    print(f"   thr={t_best:.3f}(F1최적)  F1={f1_best:.4f}  P={precision_score(y, pred_b):.4f}  R={recall_score(y, pred_b):.4f}")

    # ------------------------------------------------------------------
    # 최종 제출 파일 (fold-averaged test 예측을 동일 가중치로 블렌드)
    # ------------------------------------------------------------------
    final_test = best_w * test_lgb + (1 - best_w) * test_xgb
    sub = pd.DataFrame({"id": test["id"], TARGET: final_test})
    sub.to_csv(OUT_PATH, index=False)
    print(f"\n[제출파일] {OUT_PATH} 저장 ({len(sub)}행, 확률범위 {final_test.min():.4f}~{final_test.max():.4f})")

In [47]:
if __name__ == "__main__":
    main()

[5-Fold OOF 학습 시작] train=132027행, test=33007행
  fold 1: LightGBM PR-AUC=0.7286  XGBoost PR-AUC=0.7293
  fold 2: LightGBM PR-AUC=0.7233  XGBoost PR-AUC=0.7235
  fold 3: LightGBM PR-AUC=0.7382  XGBoost PR-AUC=0.7379
  fold 4: LightGBM PR-AUC=0.7352  XGBoost PR-AUC=0.7355
  fold 5: LightGBM PR-AUC=0.7292  XGBoost PR-AUC=0.7289

[개별 모델 OOF 전체] LightGBM=0.7306  XGBoost=0.7308

[최적 블렌드] w_LightGBM=0.430  w_XGBoost=0.570
[최종 OOF 성능] ★PR-AUC=0.7312(공식지표)  ROC-AUC=0.8895(참고)
   thr=0.50        F1=0.6363  P=0.7458  R=0.5549
   thr=0.346(F1최적)  F1=0.6663  P=0.6539  R=0.6792

[제출파일] /content/drive/MyDrive/해커톤/submission_best_stack.csv 저장 (33007행, 확률범위 0.0052~0.9797)
